In [1]:
# !pip install -q datasets

In [2]:
from datasets import load_dataset

In [3]:
dados_redacoes = load_dataset('csv', data_files='/content/redacoes.csv')

In [4]:
dados_redacoes

DatasetDict({
    train: Dataset({
        features: ['essay', 'score'],
        num_rows: 4570
    })
})

## Pelo fato da base de dados ser pequena, isso invabiliza utiliza-la para treinar o nosso modelo. Então vamos utilizar uma técnica chamada de Transferência de aprendizado, onde recuperamos o aprendizado de um modelo que foi treinado com uma grande base de dados e transferimos ao nosso modelo

In [5]:
dados_redacoes['train'].features

{'essay': Value('string'), 'score': Value('int64')}

In [6]:
dados_redacoes['train']['essay'][0]

'A convivência com o ensino em uma escola pública , nos mostra \\xa0como a reforma do ensino médio é necessária, com o ensino precário, escolas quase desabando, falta de educadores,a formação de estudantes fica a \\xa0cada dia mais difícil., Atualmente o índice de jovens que procuram uma educação de qualidade é maior do que em todos os séculos , temos em vista que todos procuram a melhoria, tanto na estrutura da instituição como no ensino, muitos sonham em ser: médicos, professores,advogados, contadores, e assim sucessivamente., A solução seria a constatação de uma lei em prol dos estudantes, precisamos ser ouvidos, a voz dos jovens é a voz do futuro, queremos o ensino superior, queremos oportunidades em escolas públicas, escolas bem estruturada e profissionais competentes., Creio \\xa0que a reforma deveria ocorrer, pois o intuito é a melhoria (Que tipo de melhoria?,\\xa0assim todos os jovens teriam a oportunidade de serem profissionais de sucesso. O incentivo ( aos jovens é o que torn

In [7]:
dados_redacoes['train']['score'][0]

4

In [8]:
dados_redacoes['train'].to_pandas()

,essay,score
0,A convivência com o ensino em uma escola públi...,4
1,O Brasil possui aproximadamente 425 mil polici...,5
2,"O esporte é muito importante, as Olimpíadas, u...",8
3,"O conceito de ""banalidade do mal"" da filósofa ...",8
4,Um dos assuntos que mais geram discussões atua...,5
...,...,...
4565,"Com o início da era digital, a capacidade de t...",6
4566,"O feminicídio é o assassinato de uma mulher, a...",4
4567,Foram vrias as conquistas que as mulheres bras...,5
4568,Segundo o psicoterapeuta Ivan Carpelatto: “a v...,9


In [9]:
treino_teste = dados_redacoes['train'].train_test_split(test_size = 0.2, shuffle = False)
treino_teste

DatasetDict({
    train: Dataset({
        features: ['essay', 'score'],
        num_rows: 3656
    })
    test: Dataset({
        features: ['essay', 'score'],
        num_rows: 914
    })
})

In [10]:
from datasets import DatasetDict

In [11]:
dados_redacoes = DatasetDict({
    'treino': treino_teste['train'],
    'teste': treino_teste['test']
})

In [12]:
dados_redacoes

DatasetDict({
    treino: Dataset({
        features: ['essay', 'score'],
        num_rows: 3656
    })
    teste: Dataset({
        features: ['essay', 'score'],
        num_rows: 914
    })
})

In [13]:
checkpoint_modelo = 'Geotrend/distilbert-base-pt-cased'

In [14]:
from transformers import AutoTokenizer

In [15]:
tokenizador = AutoTokenizer.from_pretrained(checkpoint_modelo)

In [16]:
texto = dados_redacoes['treino']['essay'][0]

In [17]:
tokens = tokenizador.tokenize(texto)
print(tokens)

['A', 'con', '##viv', '##ência', 'com', 'o', 'ensino', 'em', 'uma', 'escola', 'pública', ',', 'nos', 'mostra', '\\', 'xa', '##0', '##com', '##o', 'a', 'reforma', 'do', 'ensino', 'médio', 'é', 'ne', '##ces', '##s', '##ária', ',', 'com', 'o', 'ensino', 'pre', '##c', '##ário', ',', 'escolas', 'quase', 'desa', '##band', '##o', ',', 'falta', 'de', 'edu', '##cado', '##res', ',', 'a', 'formação', 'de', 'estudantes', 'fica', 'a', '\\', 'xa', '##0', '##cada', 'dia', 'mais', 'difícil', '.', ',', 'Atualmente', 'o', 'índice', 'de', 'jovens', 'que', 'procura', '##m', 'uma', 'educação', 'de', 'qualidade', 'é', 'maior', 'do', 'que', 'em', 'todos', 'os', 'séculos', ',', 'tem', '##os', 'em', 'vista', 'que', 'todos', 'procura', '##m', 'a', 'melhor', '##ia', ',', 'tanto', 'na', 'estrutura', 'da', 'instituição', 'como', 'no', 'ensino', ',', 'muitos', 'son', '##ham', 'em', 'ser', ':', 'médicos', ',', 'professore', '##s', ',', 'ad', '##vog', '##ados', ',', 'conta', '##dores', ',', 'e', 'assim', 'su', '##ces

In [18]:
ids = tokenizador.convert_tokens_to_ids(tokens)
print(ids)

[46, 270, 24522, 2475, 303, 91, 13849, 348, 469, 5148, 5155, 25, 1125, 4201, 73, 5308, 759, 5563, 231, 77, 8008, 246, 13849, 19640, 144, 539, 2597, 206, 4354, 25, 303, 91, 13849, 1480, 415, 2854, 25, 16687, 8774, 3151, 4948, 231, 25, 5988, 203, 5725, 8294, 920, 25, 77, 11569, 203, 22157, 13510, 77, 73, 5308, 759, 6073, 608, 578, 7859, 27, 25, 12242, 91, 20373, 203, 18423, 220, 21237, 244, 469, 17937, 203, 16944, 144, 2695, 246, 220, 348, 1692, 463, 16426, 25, 1802, 383, 348, 3416, 220, 1692, 21237, 244, 77, 5915, 359, 25, 1813, 230, 9885, 240, 21600, 314, 286, 13849, 25, 6455, 385, 1948, 348, 510, 39, 17306, 25, 17575, 206, 25, 712, 22852, 2092, 25, 5719, 2669, 25, 81, 4832, 292, 2597, 12125, 576, 27, 25, 46, 2996, 10877, 4245, 77, 10673, 15832, 203, 469, 2900, 348, 955, 258, 441, 22157, 25, 15218, 2041, 510, 465, 17157, 25, 77, 4567, 441, 18423, 144, 77, 4567, 246, 5343, 25, 15894, 16539, 91, 13849, 3121, 25, 15894, 16539, 21388, 348, 16687, 13027, 25, 16687, 4577, 9885, 317, 81, 2339

In [19]:
texto_decodificado = tokenizador.decode(ids)
print(texto_decodificado)

A convivência com o ensino em uma escola pública, nos mostra \ xa0como a reforma do ensino médio é necessária, com o ensino precário, escolas quase desabando, falta de educadores, a formação de estudantes fica a \ xa0cada dia mais difícil., Atualmente o índice de jovens que procuram uma educação de qualidade é maior do que em todos os séculos, temos em vista que todos procuram a melhoria, tanto na estrutura da instituição como no ensino, muitos sonham em ser : médicos, professores, advogados, contadores, e assim sucessivamente., A solução seria a constatação de uma lei em prol dos estudantes, precisamos ser ouvidos, a voz dos jovens é a voz do futuro, queremos o ensino superior, queremos oportunidades em escolas públicas, escolas bem estruturada e profissionais competentes., Creio \ xa0que a reforma deveria ocorrer, pois o intuito é a melhoria ( Que tipo de melhoria?, \ xa0assim todos os jovens teriam a oportunidade de serem profissionais de sucesso. O incentivo ( aos jovens é o que to

In [20]:
input_codificado = tokenizador(texto, return_tensors = 'pt')
print(input_codificado)

{'input_ids': tensor([[   11,    46,   270, 24522,  2475,   303,    91, 13849,   348,   469,
          5148,  5155,    25,  1125,  4201,    73,  5308,   759,  5563,   231,
            77,  8008,   246, 13849, 19640,   144,   539,  2597,   206,  4354,
            25,   303,    91, 13849,  1480,   415,  2854,    25, 16687,  8774,
          3151,  4948,   231,    25,  5988,   203,  5725,  8294,   920,    25,
            77, 11569,   203, 22157, 13510,    77,    73,  5308,   759,  6073,
           608,   578,  7859,    27,    25, 12242,    91, 20373,   203, 18423,
           220, 21237,   244,   469, 17937,   203, 16944,   144,  2695,   246,
           220,   348,  1692,   463, 16426,    25,  1802,   383,   348,  3416,
           220,  1692, 21237,   244,    77,  5915,   359,    25,  1813,   230,
          9885,   240, 21600,   314,   286, 13849,    25,  6455,   385,  1948,
           348,   510,    39, 17306,    25, 17575,   206,    25,   712, 22852,
          2092,    25,  5719,  2669,  

In [21]:
def funcao_tokenizadora(dados_texto):
    return tokenizador(dados_texto['essay'], truncation = True)

In [22]:
dataset_tokenizado = dados_redacoes.map(funcao_tokenizadora, batched = True, remove_columns = ['essay'])

Map:   0%|          | 0/3656 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

In [23]:
dataset_tokenizado

DatasetDict({
    treino: Dataset({
        features: ['score', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3656
    })
    teste: Dataset({
        features: ['score', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 914
    })
})

In [24]:
dataset_tokenizado['treino'].to_pandas()

,score,input_ids,token_type_ids,attention_mask
0,4,"[11, 46, 270, 24522, 2475, 303, 91, 13849, 348...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,5,"[11, 60, 1497, 6656, 5124, 9019, 2782, 22218, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,8,"[11, 60, 290, 8465, 144, 3240, 1668, 25, 243, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,8,"[11, 60, 19131, 203, 15, 23380, 19311, 246, 28...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,5,"[11, 2567, 441, 17363, 206, 220, 578, 2989, 78...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...
3651,5,"[11, 46, 5148, 1802, 469, 17870, 7326, 230, 11...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3652,6,"[11, 60, 7234, 1280, 3543, 1802, 91, 2885, 203...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3653,4,"[11, 7855, 325, 4776, 4323, 1559, 14086, 13232...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3654,6,"[11, 60, 23602, 235, 326, 23949, 206, 203, 179...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [25]:
dataset_tokenizado['teste'].to_pandas()

,score,input_ids,token_type_ids,attention_mask
0,7,"[11, 60, 9133, 18342, 2294, 220, 13606, 1295, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,8,"[11, 599, 3088, 1306, 25, 22568, 2284, 2251, 8...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,4,"[11, 46, 15182, 20233, 16866, 2969, 14150, 81,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,7,"[11, 60, 10910, 303, 469, 7592, 1828, 21261, 1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,7,"[11, 1876, 91, 3222, 240, 1774, 9151, 359, 25,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...
909,6,"[11, 3444, 91, 5764, 240, 452, 3543, 25, 77, 1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
910,4,"[11, 60, 4880, 1235, 2011, 3575, 144, 91, 2370...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
911,5,"[11, 24848, 98, 8544, 243, 6634, 206, 220, 243...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
912,9,"[11, 4433, 91, 22999, 22429, 14533, 3507, 304,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


# Ajustando a variavel Alvo

In [26]:
dataset_tokenizado['treino'].features['score']

Value('int64')

In [27]:
from datasets import ClassLabel

In [28]:
dataset_tokenizado['treino'].unique('score')

[4, 5, 8, 6, 10, 7, 1, 9, 3, 0, 2]

In [29]:
scores = ClassLabel(names = [str(i) for i in range(11)])

In [30]:
scores

ClassLabel(names=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10'])

In [31]:
def mapear_labels(dados):
    dados['label'] = scores.str2int(str(dados['score']))
    return dados

In [32]:
dataset_tokenizado = dataset_tokenizado.map(mapear_labels, remove_columns = ['score', 'token_type_ids'])

Map:   0%|          | 0/3656 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

In [33]:
dataset_tokenizado = dataset_tokenizado.cast_column('label', scores)

Casting the dataset:   0%|          | 0/3656 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/914 [00:00<?, ? examples/s]

In [34]:
dataset_tokenizado

DatasetDict({
    treino: Dataset({
        features: ['input_ids', 'attention_mask', 'label'],
        num_rows: 3656
    })
    teste: Dataset({
        features: ['input_ids', 'attention_mask', 'label'],
        num_rows: 914
    })
})

In [35]:
dataset_tokenizado['treino'].features['label']

ClassLabel(names=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10'])